# PASO 1: Importaciones y Carga del Dataset


In [ ]:
# ==============================================================================
# CELDA 1: CONFIGURACIÓN Y CARGA DE DATOS
# ==============================================================================
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import os
from google.colab import drive

# 1. Montar el disco
drive.mount('/content/drive')

# 2. Definir rutas (Asegúrate de que apunten a tu carpeta Datos_Procesados)
BASE_DIR = '/content/drive/MyDrive/CICLO_9/Integrador/Entrenamiento_GNN'
INPUT_FILE = os.path.join(BASE_DIR, 'Datos_Procesados', 'dataset_gnn_granular_final.parquet')
OUTPUT_FILE = os.path.join(BASE_DIR, 'Datos_Procesados', 'dataset_features_gnn.parquet')

# 3. Cargar el dataset granular
print("Cargando dataset base...")
df = pd.read_parquet(INPUT_FILE)
print(f"Total de registros cargados: {len(df)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cargando dataset base...
Total de registros cargados: 573527


# PASO 2: Ingeniería de Características Temporales


In [ ]:
# ==============================================================================
# CELDA 2: TRANSFORMACIÓN CÍCLICA DEL TIEMPO
# ==============================================================================
print("Aplicando transformaciones trigonométricas al tiempo...")

# 1. Extraer la hora numérica, el mes y el día de la semana
df['hora_num'] = pd.to_datetime(df['hora_delito'], format='%H:%M:%S').dt.hour
df['mes_num'] = pd.to_datetime(df['fecha_delito']).dt.month
df['dia_semana'] = pd.to_datetime(df['fecha_delito']).dt.weekday # Lunes=0, Domingo=6

# 2. Transformación Cíclica de la Hora
df['hora_sin'] = np.sin(2 * np.pi * df['hora_num'] / 24)
df['hora_cos'] = np.cos(2 * np.pi * df['hora_num'] / 24)

# 3. Transformación Cíclica del Mes
df['mes_sin'] = np.sin(2 * np.pi * df['mes_num'] / 12)
df['mes_cos'] = np.cos(2 * np.pi * df['mes_num'] / 12)

# 4. Feature Categórica: ¿Es fin de semana? (1 = Sí, 0 = No)
df['es_fin_de_semana'] = (df['dia_semana'] >= 4).astype(int)

print("Ingeniería temporal completada exitosamente.")

Aplicando transformaciones trigonométricas al tiempo...
Ingeniería temporal completada exitosamente.


# PASO 3: Creación de Nodos con K-Means (Machine Learning)

In [ ]:
# ==============================================================================
# CELDA 3: INGENIERÍA ESPACIAL (NODOS DEL GRAFO)
# ==============================================================================
print("Construyendo Nodos (Cuadrantes) del Grafo mediante Clustering...")

# Definimos cuántos Nodos (Cuadrantes) queremos en todo Lima
NUM_NODOS = 400

# Extraemos solo las coordenadas espaciales
coordenadas = df[['latitud', 'longitud']].values

# Aplicamos K-Means para agrupar los puntos en 400 nodos centrales
kmeans = KMeans(n_clusters=NUM_NODOS, random_state=42, n_init=10)
df['id_nodo'] = kmeans.fit_predict(coordenadas)

# Guardamos los centroides (las coordenadas del centro de cada nodo)
centroides = kmeans.cluster_centers_

print(f"Se generaron exitosamente {NUM_NODOS} Nodos espaciales (Hotspots).")

Construyendo Nodos (Cuadrantes) del Grafo mediante Clustering...
Se generaron exitosamente 400 Nodos espaciales (Hotspots).


# PASO 4: Exportación de la Matriz de Características

In [ ]:
# ==============================================================================
# CELDA 4: SELECCIÓN DE FEATURES Y EXPORTACIÓN
# ==============================================================================
print("Preparando exportación de la matriz de características...")

# Nos quedamos estrictamente con las features matemáticas y la referencia
columnas_finales = [
    'fecha_delito', 'id_nodo', 'tipo_delito', 'es_fin_de_semana',
    'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos',
    'latitud', 'longitud' # (Mantenidas temporalmente para validación visual)
]

df_features = df[columnas_finales]

# Guardar en formato Parquet
df_features.to_parquet(OUTPUT_FILE, index=False)

print(f"Matriz guardada en: {OUTPUT_FILE}")
print("¡Fase de Feature Engineering completada!")

Preparando exportación de la matriz de características...
Matriz guardada en: /content/drive/MyDrive/CICLO_9/Integrador/Entrenamiento_GNN/Datos_Procesados/dataset_features_gnn.parquet
¡Fase de Feature Engineering completada!


# PASO 5: VERIFICACION

In [ ]:
import pandas as pd

# 1. Leemos el archivo binario Parquet
ruta_parquet = '/content/drive/MyDrive/CICLO_9/Integrador/Entrenamiento_GNN/Datos_Procesados/dataset_features_gnn.parquet'
df_verificacion = pd.read_parquet(ruta_parquet)

# 2. Mostramos la estructura y peso real en memoria RAM
print("--- INFO DEL DATASET ---")
df_verificacion.info()

# 3. Mostramos las primeras 5 filas (Tu equivalente a abrir Excel)
print("\n--- VISTA PREVIA DE LOS DATOS ---")
display(df_verificacion.head())

--- INFO DEL DATASET ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 573527 entries, 0 to 573526
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   fecha_delito      573527 non-null  object 
 1   id_nodo           573527 non-null  int32  
 2   tipo_delito       573527 non-null  object 
 3   es_fin_de_semana  573527 non-null  int64  
 4   hora_sin          573527 non-null  float64
 5   hora_cos          573527 non-null  float64
 6   mes_sin           573527 non-null  float64
 7   mes_cos           573527 non-null  float64
 8   latitud           573527 non-null  float64
 9   longitud          573527 non-null  float64
dtypes: float64(6), int32(1), int64(1), object(2)
memory usage: 41.6+ MB

--- VISTA PREVIA DE LOS DATOS ---


,fecha_delito,id_nodo,tipo_delito,es_fin_de_semana,hora_sin,hora_cos,mes_sin,mes_cos,latitud,longitud
0,2022-01-01,10,HURTO,1,-1.000000,-1.836970e-16,0.5,0.866025,-11.732537,-77.134065
1,2022-01-25,365,HURTO,0,-1.000000,-1.836970e-16,0.5,0.866025,-11.741768,-77.160090
2,2022-01-10,365,HURTO,0,-0.965926,-2.588190e-01,0.5,0.866025,-11.737720,-77.148222
3,2022-01-08,377,HURTO,1,-0.965926,-2.588190e-01,0.5,0.866025,-11.814292,-77.131512
4,2022-01-10,246,HURTO,0,-0.707107,7.071068e-01,0.5,0.866025,-11.753594,-77.160785


# PASO 6: Construyendo la MatriZ

In [ ]:
# ==============================================================================
# CELDA 5: CONSTRUCCIÓN DEL GRAFO (MATRIZ EDGE_INDEX) - VERSIÓN ROBUSTA
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
import os

print("Calculando topología del grafo (K-Vecinos Más Cercanos)...")

# Rutas de archivo
BASE_DIR = '/content/drive/MyDrive/CICLO_9/Integrador/Entrenamiento_GNN'
INPUT_FILE = os.path.join(BASE_DIR, 'Datos_Procesados', 'dataset_features_gnn.parquet')

# 1. Recuperar los centroides desde el disco duro (A prueba de memoria vacía)
print("Recuperando nodos desde el archivo Parquet...")
df_features = pd.read_parquet(INPUT_FILE)
# Agrupamos los delitos por nodo y sacamos el promedio de sus coordenadas para hallar el "centro"
centroides = df_features.groupby('id_nodo')[['latitud', 'longitud']].mean().values

# 2. Configurar el Hiperparámetro K
# K=6 porque el algoritmo se encuentra a sí mismo como el vecino #1 (distancia 0)
K_VECINOS = 6

# 3. Entrenar el modelo espacial K-NN
knn = NearestNeighbors(n_neighbors=K_VECINOS, algorithm='ball_tree')
knn.fit(centroides)

# distances nos da la distancia, indices nos da con qué Nodos se conecta
distances, indices = knn.kneighbors(centroides)

# 4. Construir la matriz edge_index para PyTorch
fuentes = []
destinos = []
pesos = []

for i in range(len(indices)):
    for j in range(1, K_VECINOS): # Empezamos en 1 para ignorar la auto-conexión
        nodo_origen = i
        nodo_destino = indices[i][j]
        distancia = distances[i][j]

        fuentes.append(nodo_origen)
        destinos.append(nodo_destino)
        pesos.append(1.0 / (distancia + 1e-5))

# Transponer para formato PyTorch Geometric [2, N]
edge_index = np.array([fuentes, destinos])
edge_weights = np.array(pesos)

# 5. Guardar la estructura del Grafo
ruta_grafo = os.path.join(BASE_DIR, 'Datos_Procesados', 'grafo_edge_index.npz')
np.savez(ruta_grafo, edge_index=edge_index, edge_weights=edge_weights)

print(f"\n¡Grafo construido! Total de conexiones (aristas): {len(fuentes)}")
print(f"Matriz edge_index guardada en: {ruta_grafo}")

Calculando topología del grafo (K-Vecinos Más Cercanos)...
Recuperando nodos desde el archivo Parquet...

¡Grafo construido! Total de conexiones (aristas): 2000
Matriz edge_index guardada en: /content/drive/MyDrive/CICLO_9/Integrador/Entrenamiento_GNN/Datos_Procesados/grafo_edge_index.npz
